Before reading this notebook, make sure you have already gone through the **[Transformer Architecture Notebook]**, as it provides the foundational concepts needed to understand the details discussed here.

# **Introduction**
**Self-attention** is the core mechanism that revolutionized natural language processing and gave transformers their name. Unlike traditional neural networks that process sequences step-by-step, self-attention allows each position in a sequence to directly attend to and gather information from all other positions simultaneously. This mechanism computes **attention weights** that determine how much each element should focus on every other element in the sequence, enabling the model to capture **long-range dependencies** and **contextual relationships** regardless of distance. **For example,** in the sentence *"The cat that was sleeping on the mat woke up,"* self-attention helps the model understand that "woke up" relates to "cat" even though they're separated by several words. This parallel processing capability not only makes transformers more computationally efficient than recurrent networks but also allows them to build rich, contextualized representations where each word's meaning is informed by its relationship to every other word in the sequence. The "self" in self-attention refers to the fact that the queries, keys, and values used to compute attention all come from the **same input sequence**, allowing the model to discover internal structure and dependencies within the data itself.



## **Input Word Embeddings**

To use self-attention, each word in the sentence must first be converted into a dense vector using **word embeddings**.  
Take the example:  
*"The cat that was sleeping on the mat woke up."*

1. **Tokenization**  
   - Tokens: ["The", "cat", "that", "was", "sleeping", "on", "the", "mat", "woke", "up"]

2. **Embedding Matrix**  
   - We define a learnable embedding matrix:  
   $
   E \in \mathbb{R}^{V \times d}
  $  
   where $(V$) = vocabulary size and $(d$) = embedding dimension.

3. **Word to Vector Mapping**  
   - Each token index $(i$) looks up its corresponding vector in $(E$):  
 $
   x_i = E[w_i], \quad \text{for } i=1,2,\dots,n
   $
   where $(w_i$) is the index of the $(i^{th}$) word in the vocabulary.

4. **Final Input Representation**  
   - The entire sentence is then represented as a matrix of embeddings:  
  $
   X = [x_1, x_2, x_3, \dots, x_n] \in \mathbb{R}^{n \times d}
  $  
   For example, $(x_2$) (for *“cat”*) is not just a one-hot vector, but a dense vector that places it close to semantically similar words like *“dog”* or *“kitten”* in the embedding space.

Thus, word embeddings transform discrete tokens into **dense, continuous vectors**, forming the foundation for self-attention to capture contextual meaning.


Each word is first converted into an embedding vector.  
For example, if the embedding dimension is **512**, then each word is represented by a 512-dimensional vector:

- The → [512-dim vector]  
- cat → [512-dim vector]  
- sat → [512-dim vector]  

So at this stage, we have a sequence of vectors of size **(sequence_length × 512)**.

In the sentence *"The cat that was sleeping on the mat woke up."*,  
the word **"cat"** is mapped through an **embedding matrix** to a  
512-dimensional dense vector. For example:

$
\text{Embed("cat")} = [0.12, -0.48, 0.33, ..., 0.07] \in \mathbb{R}^{512}
$

Each of the 512 values is a learned floating-point number.  
Together, they encode semantic and syntactic properties of "cat",  
so that words with similar meaning lie close in this high-dimensional space.

*After that we add positional embedding which we will learn in details in next notebook.*


## Step 2: Creating Query, Key, and Value Vectors

For self-attention, we don’t directly use the embeddings. Instead, for each input embedding **E**, we create three new vectors:

- **Query (Q)**  
- **Key (K)**  
- **Value (V)**  

How? By multiplying the embedding vector by three learned weight matrices:

$
Q = E \times W^Q, \quad K = E \times W^K, \quad V = E \times W^V
$

Here:
- $(W^Q, W^K, W^V)$ are trainable matrices.

If the embedding size is 512 and we want Q, K, V to be 64-dimensional, then each \(W\) matrix will have shape **(512 × 64)**.

Thus:
- Input embedding \(E\): shape (1 × 512)  
- Query \(Q\): shape (1 × 64)  
- Key \(K\): shape (1 × 64)  
- Value \(V\): shape (1 × 64)  

So each word now has its own Q, K, V vectors.

<div align= 'center'>

[![QKV-h.png](https://i.postimg.cc/4xDz8VBy/QKV-h.png)](https://postimg.cc/SnG2R20b)

*figure: Query, Key and Value*
</div>

In the self-attention mechanism, each word embedding $(X$) is transformed into three distinct vectors: **Query (Q)**, **Key (K)**, and **Value (V)**. This transformation is achieved by multiplying the same input embedding $(X$) with three different learnable weight matrices:  

$
Q = X W^{Q}, \quad K = X W^{K}, \quad V = X W^{V}
$

Here, $(W^{Q}$), $(W^{K}$), and $(W^{V}$) are parameter matrices that define how the same input is projected into different roles.  
- **Queries (Q):** Represent what the word is “searching for” in other words.  
- **Keys (K):** Represent the “features” of each word that can be matched against queries.  
- **Values (V):** Contain the actual semantic information that will be aggregated.  

Although the input embeddings $(X$) are identical for these projections, the **difference lies in the weight matrices**. These matrices are initialized randomly (commonly with Xavier or He initialization to maintain stability in forward/backward passes) and then updated during training through **backpropagation**.  

When the model computes attention and produces an output, the loss function (e.g., cross-entropy for language modeling) compares predictions with the ground truth. The error signal is propagated backward, and gradients update $(W^{Q}$), $(W^{K}$), and $(W^{V}$). Over time, this process tunes the matrices so that queries, keys, and values align in a way that captures meaningful contextual relationships between words.  

Thus, $W^{Q}, W^{K}, W^{V}$ are not static—they **evolve dynamically during training**, shaping how the model learns to attend to relevant words and ignore irrelevant ones.



## Step 3: Self-Attention Calculation (Vector Form)

For a given word, the attention score with another word is computed by:

$
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d_k}}\right)V
$

- $(Q \cdot K^T)$ → measures similarity between the current word (Q) and other words (K).  
- $(\sqrt{d_k})$ → scaling factor (where $(d_k = 64))$, prevents values from becoming too large.  
- Softmax → converts similarity scores into probabilities (weights).  
- Multiply by \(V\) → combines information from other words, weighted by relevance.


### Why $(Q \cdot K^T $)?
 Queries $(Q$) represent **what a word is looking for** (its request for context). Keys $(K$) represent **what information a word offers**. To measure how much one word should attend to another, we compute **similarity** via the dot product:  
 Here $(d_k$) is the dimension of queries/keys, which is used to prevent from **vanishing gradient** and unstable training by scaling **softmax** values.
  

>> $
\frac{QK^T}{\sqrt{d_k}}
$

This normalization keeps values in a reasonable range and ensures stable training. It’s a **hyperparameter** that balances efficiency and expressiveness.

### Why $(QK^T$) and not $(KQ$)?
In self-attention, the **query (Q)** represents the current word asking for context, while the **key (K)** represents all words offering information. To compute how much one word should attend to another, we compare every query with every key. This requires aligning each query vector with all key vectors, which is achieved by multiplying $(Q) ((n \times d_k)$) with $(K^T) ((d_k \times n)$). The result is an $((n \times n)$) attention matrix where entry $((i,j)$) tells us *"how much word i should focus on word j."*

If we instead computed $(KQ^T$), the roles would flip: it would mean *"how much word j should focus on word i,"* reversing the direction of attention and breaking the intended meaning. That’s why the query must be on the left and the key (transposed) on the right.


###  Why transpose $(K$)?
The transpose ensures the **dimensions align** for matrix multiplication. Queries are row vectors ($(1 \times d_k$)), and keys are also row vectors. To compare them using a dot product, we need keys as column vectors ($(d_k \times 1$)). Transposing $(K$) flips it into the right orientation, so that the dot product is valid. Without the transpose, the operation would not even be mathematically consistent.


### Putting it all together
$
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$

1. $( QK^T$): compute pairwise similarities between all words.  
2. $( \sqrt{d_k} $): scale values to avoid large magnitudes.  
3. **Softmax**: convert scores into probabilities (attention weights).  
4. Multiply by $(V$): aggregate information from other words, weighted by relevance.


## Step 4: Matrix Implementation

Instead of doing this word by word, we **batch the operations**:

1. Collect all embeddings into a matrix \(E\) of shape **(sequence_length × 512)**.  
2. Multiply with learned weight matrices:  

> $
  Q = E \times W^Q \quad (sequence\_length \times 64)
  $

> $
  K = E \times W^K \quad (sequence\_length \times 64)
  $

> $
  V = E \times W^V \quad (sequence\_length \times 64)
  $

3. Compute attention scores:

$
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{64}}\right)V
$

 $(QK^T)$→ gives a matrix of shape **(sequence_length × sequence_length)**, representing pairwise similarity between words. Softmax ensures weights across each row sum to 1. Multiply with \(V\) to get the final attended representation for each word.

*We will learn this in details how matrix multiplication is done in next notebook.*



# What are $(W^Q, W^K, W^V)$ Exactly?

- They are **trainable weight matrices**, just like the weights in any neural network layer.  
- They start as **random matrices**, but during training via backpropagation, they **learn useful transformations**.  
- Their roles:  
  - $(W^Q)$: projects embeddings into Query space (what am I looking for?).  
  - $(W^K)$: projects embeddings into Key space (what do I contain?).  
  - $(W^V)$: projects embeddings into Value space (what information should I pass?).  

 Each word embedding vector is multiplied with these matrices to produce its Q, K, V vectors.


###Initialization of Weight Matrices ($(W^Q, W^K, W^V $))

- In Transformers, the matrices $(W^Q, W^K, W^V$) are **learnable parameters** that project input embeddings ($(X$)) into the **query, key, and value spaces**.  
- At the start of training, these matrices are not meaningful yet — they are **randomly initialized** using standard deep learning strategies to ensure stable gradients.


###  Common Initialization Methods

When training Transformers, the weight matrices ($(W^Q, W^K, W^V$)) must be **initialized properly** to avoid exploding or vanishing gradients. Two popular strategies are **Xavier (Glorot)** and **Kaiming (He) Initialization**.


#### 1. Xavier (Glorot) Initialization
This method ensures that the variance of activations is balanced across layers, preventing values from becoming too large or too small.  

The formula is:

$
W \sim \mathcal{U}\left(-\sqrt{\frac{6}{d_{in}+d_{out}}}, \; \sqrt{\frac{6}{d_{in}+d_{out}}}\right)
$

where
- $(W$): the weight matrix being initialized (e.g., $(W^Q, W^K, W^V$)).  
- $(\mathcal{U}(a, b)$): a uniform random distribution between $(a$) and $(b$).  
- $(d_{in}$): the input dimension (number of columns in \(W\), i.e., how many features come in).  
- $(d_{out}$): the output dimension (number of rows in \(W\), i.e., how many features go out).  
- $(\sqrt{\frac{6}{d_{in}+d_{out}}}$): scaling factor that keeps the variance consistent across layers.  

So, every entry in $(W$) is sampled from a uniform distribution in the range:  
$
\left[-\sqrt{\frac{6}{d_{in}+d_{out}}}, \; +\sqrt{\frac{6}{d_{in}+d_{out}}}\right]
$




#### 2. Kaiming (He) Initialization
This method is designed to work best with **ReLU activations** (which are common in feed-forward layers). It adjusts the scale of weights based on the input size.

$
W \sim \mathcal{N}\left(0, \; \frac{2}{d_{in}}\right)
$

where,

- $(\mathcal{N}(\mu, \sigma^2)$): a normal (Gaussian) distribution with mean $(\mu$) and variance $(\sigma^2$).  
- Mean $(\mu = 0$): weights are centered around zero.  
- Variance $(\frac{2}{d_{in}}$): ensures variance of outputs stays stable when passed through a ReLU.  
- $(d_{in}$): the number of input dimensions (size of each input vector).  

So here, each weight in $(W$) is drawn from a **normal distribution centered at 0** with variance depending on the input size.


- **Xavier Initialization** → balances variance between input and output, commonly used in attention layers.  
- **Kaiming Initialization** → accounts for ReLU activations, keeping activations well-scaled during forward propagation.  
Both methods ensure stable training by carefully controlling the distribution of initial weights.


### During Training
- These matrices are updated via **backpropagation** with gradient descent.  
- As training progresses:
  - $(W^Q$) learns how to represent "what a word is looking for."  
  - $(W^K$) learns "what information each word provides."  
  - $(W^V$) learns "what content should be passed along."  


  
$(W^Q, W^K, W^V$) start as **randomly initialized matrices** (usually Xavier). Through training, they are adjusted to capture meaningful transformations, enabling the attention mechanism to function effectively.


# Numerical Toy Example

Let’s shrink everything so we can compute by hand:

- Embedding size = **3** (instead of 512)  
- Hidden dimension for Q, K, V = **2** (instead of 64)  



### Step 1: Input word embeddings

Suppose we have **two words** in the sequence, each with a 3-d embedding:

$
E_{word1} = [1, 2, 3], \quad E_{word2} = [0, 1, 1]
$

So embedding matrix \(E\) =

$
\begin{bmatrix}
1 & 2 & 3 \\
0 & 1 & 1
\end{bmatrix}
$



### Step 2: Define trainable matrices

Initialize toy weights for Q, K, V:

$
W^Q =
\begin{bmatrix}
1 & 0 \\
0 & 1 \\
1 & 1
\end{bmatrix}, \quad
W^K =
\begin{bmatrix}
1 & 2 \\
0 & 1 \\
1 & 0
\end{bmatrix}, \quad
W^V =
\begin{bmatrix}
2 & 0 \\
0 & 2 \\
1 & 1
\end{bmatrix}
$

* Each has shape (3 × 2) because input = 3-dim, output = 2-dim.



### Step 3: Compute Q, K, V for each word

For word 1 (`[1,2,3]`):

$
Q_1 = [4, 5], \quad K_1 = [4, 4], \quad V_1 = [5, 5]
$

for word 2 (`[0,1,1]`):

$
Q_2 = [1, 2], \quad K_2 = [1, 1], \quad V_2 = [1, 3]
$



### Step 4: Build matrices Q, K, V

$
Q = \begin{bmatrix} 4 & 5 \\ 1 & 2 \end{bmatrix}, \quad
K = \begin{bmatrix} 4 & 4 \\ 1 & 1 \end{bmatrix}, \quad
V = \begin{bmatrix} 5 & 5 \\ 1 & 3 \end{bmatrix}
$



### Step 5: Compute attention scores

$
QK^T =
\begin{bmatrix}
4 & 5 \\
1 & 2
\end{bmatrix}
\cdot
\begin{bmatrix}
4 & 1 \\
4 & 1
\end{bmatrix}
=
\begin{bmatrix}
36 & 9 \\
12 & 3
\end{bmatrix}
$

Divide by $(\sqrt{d_k} = \sqrt{2} \approx 1.41)$:

$
\frac{QK^T}{\sqrt{2}} =
\begin{bmatrix}
25.46 & 6.36 \\
8.49 & 2.12
\end{bmatrix}
$


### Step 6: Apply softmax row-wise

Row 1: softmax([25.46, 6.36]) ≈ [0.999999, 0.000001]  
Row 2: softmax([8.49, 2.12]) ≈ [0.998, 0.002]  

So attention weights =

$
\begin{bmatrix}
0.999999 & 0.000001 \\
0.998 & 0.002
\end{bmatrix}
$



### Step 7: Multiply by V

$
\text{Attention}(Q,K,V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$

$
=
\begin{bmatrix}
0.999999 & 0.000001 \\
0.998 & 0.002
\end{bmatrix}
\cdot
\begin{bmatrix}
5 & 5 \\
1 & 3
\end{bmatrix}
$

Row 1 output ≈ [5, 5]  
Row 2 output ≈ [4.996, 4.996]



##  Interpretation

- Word 1 attends almost entirely to itself → output ≈ its own V.  
- Word 2 attends mostly to Word 1 → output ≈ Word 1’s V.  


# Summary

- $(W^Q, W^K, W^V)$ are trainable projection matrices that map embeddings into new spaces.  
- They start as random, but through training they learn useful transformations.  
- The Q, K, V mechanism lets words:  
  - **Q:** ask questions  
  - **K:** provide descriptions  
  - **V:** deliver information  


In practice, we don’t rely on a single **self-attention** mechanism.  
Instead, we use **Multi-Head Attention**, which combines multiple attention heads. This allows the model to capture a broader range of relationships and semantic meanings from different subspaces.  

We will explore this in detail in the next notebook: **"Multi-Head Attention"**.




# **Resources**

https://newsletter.theaiedge.io/p/understanding-the-self-attention
